In [ ]:
# rf201_composite.py from $ROOTSYS/tutorials/roofit
from ROOT import RooRealVar, RooGaussian, RooChebychev, RooArgList, RooAddPdf, RooArgSet, RooFit, kDashed, kDotted, kTRUE, kRed, TCanvas, gPad

# Setup component pdfs
# ---------------------------------------

# Declare observable x
x = RooRealVar("x", "x", 0, 10)

# Create two Gaussian PDFs g1(x,mean1,sigma) anf g2(x,mean2,sigma) and
# their parameters
mean = RooRealVar("mean", "mean of gaussians", 5)
sigma1 = RooRealVar("sigma1", "width of gaussians", 0.5)
sigma2 = RooRealVar("sigma2", "width of gaussians", 1)

sig1 = RooGaussian("sig1", "Signal component 1", x, mean, sigma1)
sig2 = RooGaussian("sig2", "Signal component 2", x, mean, sigma2)

# Build Chebychev polynomial pdf
a0 = RooRealVar("a0", "a0", 0.5, 0., 1.)
a1 = RooRealVar("a1", "a1", -0.2, 0., 1.)
bkg = RooChebychev("bkg", "Background", x, RooArgList(a0, a1))


# Method 1 - Two RooAddPdfs
# ------------------------------------------
# Add signal components

# Sum the signal components into a composite signal pdf
sig1frac = RooRealVar(
    "sig1frac", "fraction of component 1 in signal", 0.8, 0., 1.)
sig = RooAddPdf("sig", "Signal", RooArgList(
    sig1, sig2), RooArgList(sig1frac))

# Add signal and background
# ------------------------------------------------

# Sum the composite signal and background
bkgfrac = RooRealVar("bkgfrac", "fraction of background", 0.5, 0., 1.)
model = RooAddPdf(
    "model", "g1+g2+a", RooArgList(bkg, sig), RooArgList(bkgfrac))

# Sample, fit and plot model
# ---------------------------------------------------

# Generate a data sample of 1000 events in x from model
data = model.generate(RooArgSet(x), 1000)

# Fit model to data
model.fitTo(data)

# Plot data and PDF overlaid
xframe = x.frame(RooFit.Title(
    "Example of composite pdf=(sig1+sig2)+bkg"))
data.plotOn(xframe)
model.plotOn(xframe)

# Overlay the background component of model with a dashed line
ras_bkg = RooArgSet(bkg)
model.plotOn(xframe, RooFit.Components(ras_bkg),
             RooFit.LineStyle(kDashed))

# Overlay the background+sig2 components of model with a dotted line
ras_bkg_sig2 = RooArgSet(bkg, sig2)
model.plotOn(xframe, RooFit.Components(ras_bkg_sig2),
             RooFit.LineStyle(kDotted))

# Print structure of composite pdf
model.Print("t")

# Method 2 - One RooAddPdf with recursive fractions
# ---------------------------------------------------

# Construct sum of models on one go using recursive fraction interpretations
#
#   model2 = bkg + (sig1 + sig2)
#
model2 = RooAddPdf(
    "model",
    "g1+g2+a",
    RooArgList(
        bkg,
        sig1,
        sig2),
    RooArgList(
        bkgfrac,
        sig1frac),
    kTRUE)

# NB: Each coefficient is interpreted as the fraction of the
# left-hand component of the i-th recursive sum, i.e.
#
#   sum4 = A + ( B + ( C + D)  with fraction fA, and fC expands to
#
#   sum4 = fA*A + (1-fA)*(fB*B + (1-fB)*(fC*C + (1-fC)*D))

# Plot recursive addition model
# ---------------------------------------------------------
model2.plotOn(xframe, RooFit.LineColor(kRed),
              RooFit.LineStyle(kDashed))
model2.plotOn(
    xframe,
    RooFit.Components(ras_bkg_sig2),
    RooFit.LineColor(
        kRed),
    RooFit.LineStyle(
        kDashed))
model2.Print("t")

# Draw the frame on the canvas
c = TCanvas("rf201_composite", "rf201_composite", 600, 600)
gPad.SetLeftMargin(0.15)
xframe.GetYaxis().SetTitleOffset(1.4)
xframe.Draw()

c.SaveAs("rf201_composite.png")

In [ ]:
c.Draw()